In [50]:
import json
from transformers import BertTokenizer, BertModel
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from random import choice


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
bert_dir = '/Users/bowie/Documents/muti-model/bert-base-chinese'

In [43]:
rel_path = 'rel.json'

with open(rel_path, 'r', encoding='utf-8') as f:
    rel_data = json.load(f)
    
id2rel = {int(i): rel for i, rel in rel_data.items()}
print(len(id2rel), id2rel)
rel2id = {rel: int(i) for i, rel in rel_data.items()}
print(rel2id)

18 {0: '出品公司', 1: '国籍', 2: '出生地', 3: '民族', 4: '出生日期', 5: '毕业院校', 6: '歌手', 7: '所属专辑', 8: '作词', 9: '作曲', 10: '连载网站', 11: '作者', 12: '出版社', 13: '主演', 14: '导演', 15: '编剧', 16: '上映时间', 17: '成立日期'}
{'出品公司': 0, '国籍': 1, '出生地': 2, '民族': 3, '出生日期': 4, '毕业院校': 5, '歌手': 6, '所属专辑': 7, '作词': 8, '作曲': 9, '连载网站': 10, '作者': 11, '出版社': 12, '主演': 13, '导演': 14, '编剧': 15, '上映时间': 16, '成立日期': 17}


In [4]:
# 加载预训练的 BERT tokenizer
tokenizer = BertTokenizer.from_pretrained(bert_dir)

/Users/bowie/anaconda3/envs/good/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [5]:

# 示例文本
text = "CCTV7 4月15号"

# 使用 tokenizer 编码文本
encoded = tokenizer(text, return_tensors='pt')

# 输出编码后的结果
print(encoded)

{'input_ids': tensor([[  101, 10099,  8161,   125,  3299,  8115,  1384,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}


In [31]:

def preprocess_data(data, tokenizer, max_len=300):
    tokens_batch, segments_batch = [], []
    sub_heads_batch, sub_tails_batch = [], []
    obj_heads_batch, obj_tails_batch = [], []

    for sample in data:
        text = sample['text']
        spo_list = sample['spo_list']  # 三元组 (subject, predicate, object)

        # 对文本进行编码
        encoding = tokenizer(text, return_tensors='pt', padding='max_length', truncation=True, max_length=max_len)
        input_ids = encoding['input_ids'].squeeze(0)  # 去掉批次维度
        token_type_ids = encoding['token_type_ids'].squeeze(0)

        tokens_batch.append(input_ids)
        segments_batch.append(token_type_ids)

        # 初始化subject和object的头尾标签
        sub_heads = torch.zeros(max_len)
        sub_tails = torch.zeros(max_len)
        obj_heads = torch.zeros((max_len, len(tokenizer.vocab)))
        obj_tails = torch.zeros((max_len, len(tokenizer.vocab)))

        # 遍历 spo_list 进行三元组标注
        for spo in spo_list:
            subject = spo['subject']
            obj = spo['object']
            predicate = spo['predicate']

            # 获取 subject 和 object 在文本中的位置
            subject_tokens = tokenizer(subject, return_tensors='pt')['input_ids'][0][1:-1]  # 不包括CLS和SEP
            object_tokens = tokenizer(obj, return_tensors='pt')['input_ids'][0][1:-1]

            # 在编码后的文本中查找 subject 和 object 的位置
            sub_start = (input_ids == subject_tokens[0]).nonzero(as_tuple=True)[0]
            sub_end = sub_start + len(subject_tokens) - 1

            obj_start = (input_ids == object_tokens[0]).nonzero(as_tuple=True)[0]
            obj_end = obj_start + len(object_tokens) - 1

            # 标注 subject 和 object 的头尾
            sub_heads[sub_start] = 1
            sub_tails[sub_end] = 1
            obj_heads[obj_start] = 1
            obj_tails[obj_end] = 1

        sub_heads_batch.append(sub_heads)
        sub_tails_batch.append(sub_tails)
        obj_heads_batch.append(obj_heads)
        obj_tails_batch.append(obj_tails)

    return {
        'tokens_batch': torch.stack(tokens_batch),
        'segments_batch': torch.stack(segments_batch),
        'sub_heads_batch': torch.stack(sub_heads_batch),
        'sub_tails_batch': torch.stack(sub_tails_batch),
        'obj_heads_batch': torch.stack(obj_heads_batch),
        'obj_tails_batch': torch.stack(obj_tails_batch)
    }


In [16]:
data = [
    {
        'text': "CCTV7 4月15号",
        'spo_list': [
            {'subject': 'CCTV7', 'predicate': '播出', 'object': '4月15号'}
        ]
    }
]

processed_data = preprocess_data(data, tokenizer, max_len=30)
print(processed_data)


{'tokens_batch': tensor([[  101, 10099,  8161,   125,  3299,  8115,  1384,   102,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0]]), 'segments_batch': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0]]), 'sub_heads_batch': tensor([[0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]]), 'sub_tails_batch': tensor([[0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]]), 'obj_heads_batch': tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]]]), 'obj_tails_b

In [26]:
bert = BertModel.from_pretrained(bert_dir)

A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Pl

In [39]:
class CasRelModel(nn.Module):
    def __init__(self, hidden_size=768):
        super(CasRelModel, self).__init__()
        # 加载预训练的 BERT 模型
        self.bert = bert

        # 主体抽取器：用于识别主体的头部和尾部
        self.subject_head_extractor = nn.Linear(hidden_size, 1)
        self.subject_tail_extractor = nn.Linear(hidden_size, 1)

        # 宾语抽取器：基于每个主体来识别宾语的头部和尾部
        # self.object_head_extractor = nn.Linear(hidden_size, 1)
        # self.object_tail_extractor = nn.Linear(hidden_size, 1)
        self.object_head_extractor = nn.Linear(hidden_size, tokenizer.vocab_size)
        self.object_tail_extractor = nn.Linear(hidden_size, tokenizer.vocab_size)

        # 激活函数：sigmoid 用于输出概率
        self.sigmoid = nn.Sigmoid()

    def forward(self, input_ids, attention_mask):
        # 输入文本通过 BERT 编码
        bert_output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = bert_output.last_hidden_state

        # 主体头部预测
        subject_head_logits = self.subject_head_extractor(sequence_output)
        subject_head_logits = self.sigmoid(subject_head_logits).squeeze(-1)

        # 主体尾部预测
        subject_tail_logits = self.subject_tail_extractor(sequence_output)
        subject_tail_logits = self.sigmoid(subject_tail_logits).squeeze(-1)

        # 宾语头部预测
        object_head_logits = self.object_head_extractor(sequence_output)
        object_head_logits = self.sigmoid(object_head_logits).squeeze(-1)

        # 宾语尾部预测
        object_tail_logits = self.object_tail_extractor(sequence_output)
        object_tail_logits = self.sigmoid(object_tail_logits).squeeze(-1)

        return subject_head_logits, subject_tail_logits, object_head_logits, object_tail_logits


In [29]:
def train_model(model, data_loader, optimizer, epochs=3):
    model.to(device)
    model.train()

    for epoch in range(epochs):
        total_loss = 0
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            sub_heads = batch['sub_heads'].to(device)
            sub_tails = batch['sub_tails'].to(device)
            obj_heads = batch['obj_heads'].to(device)
            obj_tails = batch['obj_tails'].to(device)

            # 清空梯度
            optimizer.zero_grad()

            # 前向传播
            sub_head_logits, sub_tail_logits, obj_head_logits, obj_tail_logits = model(input_ids, attention_mask)

            # 计算损失：二元交叉熵损失
            loss_sub_head = F.binary_cross_entropy(sub_head_logits, sub_heads)
            loss_sub_tail = F.binary_cross_entropy(sub_tail_logits, sub_tails)
            loss_obj_head = F.binary_cross_entropy(obj_head_logits, obj_heads)
            loss_obj_tail = F.binary_cross_entropy(obj_tail_logits, obj_tails)

            loss = loss_sub_head + loss_sub_tail + loss_obj_head + loss_obj_tail

            # 反向传播
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch + 1}/{epochs}, Loss: {total_loss / len(data_loader)}")

    print("Training complete.")


In [35]:


class CasRelDataset(Dataset):
    def __init__(self, data_path, tokenizer, max_len=300):
        # self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.data = []
        with open(data_path, 'r', encoding='utf-8') as f:
            for i in f:
                data_line = json.loads(i)
                self.data.append(data_line)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        text = sample['text']
        spo_list = sample['spo_list']

        # 调用之前的数据预处理函数
        processed_sample = preprocess_data([sample], self.tokenizer, self.max_len)
        
        return {
            'input_ids': processed_sample['tokens_batch'][0],
            'attention_mask': processed_sample['segments_batch'][0],
            'sub_heads': processed_sample['sub_heads_batch'][0],
            'sub_tails': processed_sample['sub_tails_batch'][0],
            'obj_heads': processed_sample['obj_heads_batch'][0],
            'obj_tails': processed_sample['obj_tails_batch'][0],
        }


In [36]:
dev_data_path = 'dev.json'
# 加载数据集
dev_dataset = CasRelDataset(dev_data_path, tokenizer)
dev_data_loader = DataLoader(dev_dataset, batch_size=4, shuffle=True)


In [40]:
# 初始化模型和优化器
model = CasRelModel()
optimizer = optim.Adam(model.parameters(), lr=3e-5)


In [41]:
# 训练模型
train_model(model, dev_data_loader, optimizer, epochs=3)


KeyboardInterrupt: 

## 第二种

In [46]:
class E2EModel(nn.Module):
    def __init__(self, num_rels):
        super(E2EModel, self).__init__()
        self.bert = bert

        # 主语的头尾预测
        self.sub_head_pred = nn.Linear(self.bert.config.hidden_size, 1)
        self.sub_tail_pred = nn.Linear(self.bert.config.hidden_size, 1)
        # 客体的头尾预测
        self.obj_head_pred = nn.Linear(self.bert.config.hidden_size, num_rels)
        self.obj_tail_pred = nn.Linear(self.bert.config.hidden_size, num_rels)

    def forward_subject(self, tokens, segments):
        attention_mask = (tokens > 0).float()
        outputs = self.bert(input_ids=tokens, token_type_ids=segments, attention_mask=attention_mask)
        sequence_output = outputs[0]
        pred_sub_heads = torch.sigmoid(self.sub_head_pred(sequence_output))
        pred_sub_tails = torch.sigmoid(self.sub_tail_pred(sequence_output))
        return pred_sub_heads, pred_sub_tails

    def forward_object(self, tokens, segments, sub_head, sub_tail):
        attention_mask = (tokens > 0).float()
        outputs = self.bert(input_ids=tokens, token_type_ids=segments, attention_mask=attention_mask)
        sequence_output = outputs[0]

        sub_head_feature = sequence_output.gather(1, sub_head.unsqueeze(1).unsqueeze(2).expand(-1, -1, sequence_output.size(-1)))
        sub_tail_feature = sequence_output.gather(1, sub_tail.unsqueeze(1).unsqueeze(2).expand(-1, -1, sequence_output.size(-1)))
        sub_feature = (sub_head_feature + sub_tail_feature) / 2

        sequence_output = sequence_output + sub_feature

        pred_obj_heads = torch.sigmoid(self.obj_head_pred(sequence_output))
        pred_obj_tails = torch.sigmoid(self.obj_tail_pred(sequence_output))
        return pred_obj_heads, pred_obj_tails




In [47]:
# 训练模型可以分别初始化：
subject_model = E2EModel(len(id2rel)).forward_subject
object_model = E2EModel(len(id2rel)).forward_object
hbt_model = E2EModel(len(id2rel))  # 用于整体训练

In [48]:
def find_head_idx(source, target):
    target_len = len(target)
    for i in range(len(source)):
        if source[i: i + target_len] == target:
            return i
    return -1

In [58]:

class CustomDataset(Dataset):
    def __init__(self, data_path, tokenizer, rel2id, num_rels, maxlen):
        # self.data = data
        self.tokenizer = tokenizer
        self.rel2id = rel2id
        self.num_rels = num_rels
        self.maxlen = maxlen

        self.data = []
        with open(data_path, 'r', encoding='utf-8') as f:
            for i in f:
                data_line = json.loads(i)
                self.data.append(data_line)
        
        self.data = self.data[:1000]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        line = self.data[idx]
        text = ' '.join(line['text'].split()[:self.maxlen])
        tokens = self.tokenizer.tokenize(text)
        if len(tokens) > self.maxlen:
            tokens = tokens[:self.maxlen]

        text_len = len(tokens)
        s2ro_map = {}
        for triple in line['spo_list']:
            triple = (
                self.tokenizer.tokenize(triple[0])[1:-1], 
                triple[1], 
                self.tokenizer.tokenize(triple[2])[1:-1]
            )
            sub_head_idx = find_head_idx(tokens, triple[0])
            obj_head_idx = find_head_idx(tokens, triple[2])
            if sub_head_idx != -1 and obj_head_idx != -1:
                sub = (sub_head_idx, sub_head_idx + len(triple[0]) - 1)
                if sub not in s2ro_map:
                    s2ro_map[sub] = []
                s2ro_map[sub].append(
                    (obj_head_idx, obj_head_idx + len(triple[2]) - 1, self.rel2id[triple[1]])
                )

        if s2ro_map:
            token_ids, segment_ids = self.tokenizer.encode(first=text)
            if len(token_ids) > text_len:
                token_ids = token_ids[:text_len]
                segment_ids = segment_ids[:text_len]

            sub_heads, sub_tails = np.zeros(text_len), np.zeros(text_len)
            for s in s2ro_map:
                sub_heads[s[0]] = 1
                sub_tails[s[1]] = 1

            sub_head, sub_tail = choice(list(s2ro_map.keys()))

            obj_heads = np.zeros((text_len, self.num_rels))
            obj_tails = np.zeros((text_len, self.num_rels))
            for ro in s2ro_map.get((sub_head, sub_tail), []):
                obj_heads[ro[0]][ro[2]] = 1
                obj_tails[ro[1]][ro[2]] = 1

            return {
                'tokens': token_ids,
                'segments': segment_ids,
                'sub_heads': sub_heads,
                'sub_tails': sub_tails,
                'sub_head': sub_head,
                'sub_tail': sub_tail,
                'obj_heads': obj_heads,
                'obj_tails': obj_tails
            }


In [59]:

# Padding function
def seq_padding(inputs, padding_value=0):
    max_len = max([len(x) for x in inputs])
    return np.array([
        np.pad(x, (0, max_len - len(x)), mode='constant', constant_values=padding_value)
        for x in inputs
    ])

# Collate function to handle padding for batches
def collate_fn(batch):
    tokens_batch = seq_padding([item['tokens'] for item in batch])
    segments_batch = seq_padding([item['segments'] for item in batch])
    sub_heads_batch = seq_padding([item['sub_heads'] for item in batch])
    sub_tails_batch = seq_padding([item['sub_tails'] for item in batch])
    sub_head_batch = np.array([item['sub_head'] for item in batch])
    sub_tail_batch = np.array([item['sub_tail'] for item in batch])
    obj_heads_batch = seq_padding([item['obj_heads'] for item in batch], np.zeros((batch[0]['obj_heads'].shape[1],)))
    obj_tails_batch = seq_padding([item['obj_tails'] for item in batch], np.zeros((batch[0]['obj_tails'].shape[1],)))

    return (
        torch.tensor(tokens_batch), 
        torch.tensor(segments_batch), 
        torch.tensor(sub_heads_batch), 
        torch.tensor(sub_tails_batch), 
        torch.tensor(sub_head_batch), 
        torch.tensor(sub_tail_batch), 
        torch.tensor(obj_heads_batch), 
        torch.tensor(obj_tails_batch)
    )



In [60]:
# DataLoader
train_dataset = CustomDataset('train.json', tokenizer, rel2id, len(rel2id), 300)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)


In [56]:
def compute_loss(pred_sub_heads, pred_sub_tails, pred_obj_heads, pred_obj_tails, 
                 true_sub_heads, true_sub_tails, true_obj_heads, true_obj_tails, mask):
    """
    计算主语和客体的头尾损失
    
    Args:
        pred_sub_heads: 预测的主语头 (batch_size, seq_len, 1)
        pred_sub_tails: 预测的主语尾 (batch_size, seq_len, 1)
        pred_obj_heads: 预测的客体头 (batch_size, seq_len, num_rels)
        pred_obj_tails: 预测的客体尾 (batch_size, seq_len, num_rels)
        true_sub_heads: 真实的主语头 (batch_size, seq_len)
        true_sub_tails: 真实的主语尾 (batch_size, seq_len)
        true_obj_heads: 真实的客体头 (batch_size, seq_len, num_rels)
        true_obj_tails: 真实的客体尾 (batch_size, seq_len, num_rels)
        mask: 掩码 (batch_size, seq_len)，用于忽略padding部分
    
    Returns:
        loss: 总损失
    """
    # 计算主语头尾的二值交叉熵损失
    sub_heads_loss = F.binary_cross_entropy(pred_sub_heads.squeeze(-1), true_sub_heads, reduction='none')
    sub_tails_loss = F.binary_cross_entropy(pred_sub_tails.squeeze(-1), true_sub_tails, reduction='none')
    
    # 计算客体头尾的二值交叉熵损失
    obj_heads_loss = F.binary_cross_entropy(pred_obj_heads, true_obj_heads, reduction='none')
    obj_tails_loss = F.binary_cross_entropy(pred_obj_tails, true_obj_tails, reduction='none')

    # 使用mask忽略padding部分
    sub_heads_loss = (sub_heads_loss * mask).sum() / mask.sum()
    sub_tails_loss = (sub_tails_loss * mask).sum() / mask.sum()
    obj_heads_loss = (obj_heads_loss * mask.unsqueeze(-1)).sum() / mask.sum()
    obj_tails_loss = (obj_tails_loss * mask.unsqueeze(-1)).sum() / mask.sum()

    # 总损失为主语头尾损失和客体头尾损失之和
    total_loss = (sub_heads_loss + sub_tails_loss) + (obj_heads_loss + obj_tails_loss)
    
    return total_loss


In [ ]:
optimizer = torch.optim.Adam(hbt_model.parameters(), lr=3e-5)

for epoch in range(1):
    hbt_model.train()
    for batch in train_loader:
        tokens, segments, gold_sub_heads, gold_sub_tails, sub_head, sub_tail, gold_obj_heads, gold_obj_tails = batch
        
        # 预测
        pred_sub_heads, pred_sub_tails = subject_model(tokens, segments)
        pred_obj_heads, pred_obj_tails = object_model(tokens, segments, sub_head, sub_tail)

        # 计算损失
        loss = compute_loss(pred_sub_heads, pred_sub_tails, pred_obj_heads, pred_obj_tails,
                            gold_sub_heads, gold_sub_tails, gold_obj_heads, gold_obj_tails,
                            mask=(tokens > 0).float())

        # 反向传播和优化
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # 评估与保存模型（替代callbacks）
    if epoch % 10 == 0:  # 定期评估与保存
        evaluator.evaluate(dev_data)  # 假设你定义了Evaluate类
        torch.save(hbt_model.state_dict(), save_weights_path)


In [61]:
for epoch in range(1):
    hbt_model.train()
    for batch in train_loader:
        tokens_batch, segments_batch, sub_heads_batch, sub_tails_batch, sub_head_batch, sub_tail_batch, obj_heads_batch, obj_tails_batch = batch
        
        # 将数据传入模型
        pred_sub_heads, pred_sub_tails = hbt_model.forward_subject(tokens_batch, segments_batch)
        pred_obj_heads, pred_obj_tails = hbt_model.forward_object(tokens_batch, segments_batch, sub_head_batch, sub_tail_batch)

        # 计算损失并优化
        loss = compute_loss(pred_sub_heads, pred_sub_tails, pred_obj_heads, pred_obj_tails,
                            sub_heads_batch, sub_tails_batch, obj_heads_batch, obj_tails_batch,
                            mask=(tokens_batch > 0).float())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        print(f"Epoch {epoch}, Loss: {loss.item()}")


KeyError: 0